# picoNewton_v4 — Google Colab workflow

Steps 1–6: source validation, standalone package tests, six-artery hydrodynamic reproduction, uncoupled Piezo1 source-model verification, and passive membrane–cortex interface validation. No force-to-Piezo1 coupling is executed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPOSITORY = 'https://github.com/khalid-saqr/picoNewton.git'
BRANCH = 'agent/piconewton-v4-step6'
PROFILE = 'quick'  # use 'publication' for Step 4 N=150, Nt=2048, Nq=256
!rm -rf /content/picoNewton
!git clone --branch "$BRANCH" --single-branch "$REPOSITORY" /content/picoNewton
%cd /content/picoNewton
!python -m pip install -e './picoNewton_v4[dev]'

In [ ]:
from pathlib import Path
from piconewton_v4.runtime import create_runtime
from piconewton_v4.provenance import environment_record, write_json
drive_runs = Path('/content/drive/MyDrive/picoNewton_v4_runs')
runtime = create_runtime(
    drive_root=drive_runs,
    local_base=Path('/content/piconewton_v4_runtime'),
    metadata={'profile': PROFILE, 'branch': BRANCH, 'completed_through_step': 6},
)
write_json(runtime.manifests / 'environment.json', environment_record(Path('/content/picoNewton')))
runtime.as_dict()

In [ ]:
import json
from piconewton_v4.sources import validate_cellml_sources
source_report = validate_cellml_sources(Path('/content/picoNewton/picoNewton_v4'))
print(json.dumps(source_report, indent=2))
assert source_report['ok'], source_report

In [ ]:
!pytest /content/picoNewton/picoNewton_v4/tests

In [ ]:
from piconewton_v4.workflow_step4 import run_step4
local_step4 = runtime.local_root / f'step4_{PROFILE}'
step4 = run_step4(
    package_root=Path('/content/picoNewton/picoNewton_v4'),
    output_root=local_step4,
    profile=PROFILE,
)
print(json.dumps(step4['validation'], indent=2))
assert step4['status'] == 'passed', step4

In [ ]:
from piconewton_v4.workflow_step5 import run_step5
local_step5 = runtime.local_root / 'step5_source_model'
step5 = run_step5(
    package_root=Path('/content/picoNewton/picoNewton_v4'),
    output_root=local_step5,
)
print(json.dumps(step5, indent=2))
assert step5['status'] == 'passed', step5
assert step5['coupling_executed'] is False

In [ ]:
from piconewton_v4.workflow_step6 import run_step6
local_step6 = runtime.local_root / f'step6_{PROFILE}'
step6 = run_step6(
    package_root=Path('/content/picoNewton/picoNewton_v4'),
    output_root=local_step6,
    profile=PROFILE,
    hydrodynamic_root=local_step4,
)
print(json.dumps(step6['validation'], indent=2))
assert step6['status'] == 'passed', step6
assert step6['validation']['piezo1_coupling_executed'] is False

In [ ]:
import shutil
persistent_step4 = runtime.source_data / f'step4_{PROFILE}'
persistent_step5 = runtime.validation / 'step5_source_model'
persistent_step6 = runtime.validation / f'step6_{PROFILE}'
shutil.copytree(local_step4, persistent_step4, dirs_exist_ok=False)
shutil.copytree(local_step5, persistent_step5, dirs_exist_ok=False)
shutil.copytree(local_step6, persistent_step6, dirs_exist_ok=False)
print(persistent_step4)
print(persistent_step5)
print(persistent_step6)

In [ ]:
import pandas as pd
membrane_summary = pd.read_csv(local_step6 / 'six_artery_membrane_summary.csv')
membrane_summary[membrane_summary['pathway'].isin(['wss_anisotropic', 'force_signed_anisotropic'])][
    ['artery_id', 'pathway', 'stress_rms_pa', 'strain_rms', 'strain_peak_abs']
]

In [ ]:
from datetime import datetime, timezone
write_json(
    runtime.manifests / 'step6_completion.json',
    {
        'step': 6,
        'status': 'passed',
        'completed_utc': datetime.now(timezone.utc).isoformat(),
        'profile': PROFILE,
        'step4_output': str(persistent_step4),
        'step5_output': str(persistent_step5),
        'step6_output': str(persistent_step6),
        'piezo1_coupling_executed': False,
    },
)
print(runtime.persistent_root)